# Burn cost demo: load the baseline data

Run `python setup_demo.py` once from the project directory first.
Read the baseline snapshot from local SQL Server and save the rows for 02 and 03.
This notebook does not fit or publish a model.


In [ ]:
DATABASE_MODE = "remote"  # "local" or "remote"
RUNTIME_MODULE = "demo_sql_runtime"  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = "PricingNotebookDemo"
ALLOW_REMOTE_WRITES = False

REPLACE_DATASET = True  # Replace this demo's saved dataset when rerunning.

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from pricing_pipeline.notebook import PricingDataset, connect

MODEL_DIR = PROJECT_ROOT / "pricing_models/burn_cost_demo"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"

## Connect


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

## Read the baseline snapshot from SQL

The setup script created two synthetic snapshots. Start with 31 August. The weekly runner reads the later snapshot.
Feature transforms belong in the source SQL; this example needs none.


In [ ]:
from sqlalchemy import text

BASELINE_AS_OF = "2026-08-31"
with pricing.engine.connect() as connection:
    raw_df = pd.read_sql_query(
        text("""
            SELECT policy_id, as_of, region, bonus_malus, driver_age, exposure, burn_cost
            FROM dbo.DEMO_BURN_COST_SOURCE
            WHERE as_of = :as_of
            ORDER BY policy_id
        """),
        connection,
        params={"as_of": BASELINE_AS_OF},
    )
display({"Rows": len(raw_df), "Source date": raw_df["as_of"].unique().tolist()})

## Prepare the dataset

Keep the source date and row keys. Both 02 and 03 will use these same rows.

In [ ]:
df = raw_df.sort_values("policy_id").reset_index(drop=True)
display(df.head())

## Record provenance and save

This saves the source rows and their provenance for 02 and 03.
Replacement is enabled for this demo, so rerunning can update the local dataset. It does not change the source SQL table.


In [ ]:
dataset = PricingDataset(
    df=df,
    name="demo_burn_cost",
    source="dbo.DEMO_BURN_COST_SOURCE",
    key="policy_id",
    as_of="as_of",
)
dataset.save(DATASET_PATH, replace=REPLACE_DATASET)
print(f"Saved {len(df)} rows to {DATASET_PATH}")

Next: **02_model_exploration.ipynb** defines and fits the model.